In [ ]:
import subprocess, os, shutil

REPO_URL = "https://github.com/safety-research/legibility.git"
REPO_DIR = "/workspace/18-4-2026"
EXP_DIR = os.path.join(REPO_DIR, "experiments", "2026", "15-4-2026")

# Clone or pull latest (fetch + reset to ensure we have the newest commit)
if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)

# Install git-lfs if not available, then pull LFS files
if shutil.which("git-lfs") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(["apt-get", "install", "-y", "-qq", "git-lfs"], check=False)
    subprocess.run(["git", "lfs", "install"], check=False)
subprocess.run(["git", "-C", REPO_DIR, "lfs", "pull"], check=False)

# Install dependencies
req_path = os.path.join(EXP_DIR, "requirements.txt")
if os.path.exists(req_path):
    subprocess.run(["pip", "install", "-q", "-r", req_path], check=True)
else:
    print(f"WARNING: {req_path} not found, skipping pip install")

# Set working directory so Path.cwd().parent resolves to experiment root
os.chdir(os.path.join(EXP_DIR, "notebooks"))

From https://github.com/safety-research/legibility
   1662360..a6d0564  main       -> origin/main


HEAD is now at a6d0564 Update NB02b R5 extraction and phase2_utils


In [2]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from phase2_utils import ACTIVATIONS_DIR

r5_outputs = ["R5_last_token", "R5_cot_boundary"]
r5_all_cached = all((ACTIVATIONS_DIR / name / "metadata.json").exists() for name in r5_outputs)

if r5_all_cached:
    print("CACHED: All R5 outputs exist. Delete activations/R5_* directories to recompute.")
    for subdir in r5_outputs:
        path = ACTIVATIONS_DIR / subdir
        total_size = sum(f.stat().st_size for f in path.rglob('*') if f.is_file())
        print(f"  {subdir}: {total_size / 1e9:.2f} GB")
else:
    print("No cached R5 outputs found -- will extract from scratch.")

No cached R5 outputs found -- will extract from scratch.
No cached R5 outputs found -- will extract from scratch.


# NB02b: Reader R5 Activation Extraction (H200 GPU)

Load R5 (Gemma-4-31B-IT, full bf16) on H200 and extract activations
during C2 crossfill. This enables Experiment D: reader-side activation analysis
with a second reader from the Google model family.

**Outputs:**
- `activations/R5_last_token/` -- last-token acts at every 4th layer
- `activations/R5_cot_boundary/` -- activation at the prefill/generation boundary

**GPU time:** ~3 hours

**Note:** R5 at 31B fits on H200 at full bf16 precision (~62 GB VRAM),
giving higher-fidelity activations than R2's 4-bit quantization.

In [3]:
import sys
import json
import gc
import torch
import numpy as np
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from phase2_utils import (
    load_phase1_results, join_cots_with_labels, load_model,
    extract_activations, save_activations, load_activations,
    get_default_layer_indices, print_phase1_summary,
    LOCAL_MODELS, ACTIVATIONS_DIR,
)

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: NVIDIA H200
VRAM: 150.0 GB


In [4]:
print_phase1_summary()

# For reader analysis, we need CoTs paired with their C2 outcome.
# Load all classified CoTs (all generators, all labels except FILTERED).
all_cots = join_cots_with_labels()
print(f"Total classified CoTs with text: {len(all_cots)}")

# Build R5 crossfill prompt format
r5_inputs = []
r5_labels = []  # 1 = C2 success (reader answered correctly), 0 = C2 failure
r5_metadata = []

results = load_phase1_results()
for rec in all_cots:
    # Get R5's C2 result for this CoT
    c2_results = rec.get('c2_results', {})
    r5_correct = c2_results.get('R5', None)
    if r5_correct is None:
        continue

    # Build crossfill input: question + CoT reasoning
    crossfill_input = (
        f"{rec['input']}\n\n"
        f"Here is a reasoning trace from another model:\n"
        f"<think>\n{rec['cot_text']}\n</think>\n\n"
        f"Based on the above reasoning, what is the answer?"
    )

    r5_inputs.append(crossfill_input)
    r5_labels.append(int(r5_correct))
    r5_metadata.append({
        'sample_id': rec['sample_id'],
        'generator_id': rec['generator_id'],
        'epoch': rec['epoch'],
        'label': rec['label'],
        'r5_correct': r5_correct,
    })

print(f"R5 crossfill inputs: {len(r5_inputs)}")
print(f"R5 C2 success rate: {sum(r5_labels)/len(r5_labels):.1%}")

# Check by legibility class
from collections import Counter
for label in ['ANSWER_LEAKED', 'REASONING_LEGIBLE', 'ILLEGIBLE']:
    subset = [m for m in r5_metadata if m['label'] == label]
    if subset:
        success = sum(1 for m in subset if m['r5_correct'])
        print(f"  {label}: {len(subset)} samples, R5 success={success/len(subset):.1%}")

Total records: 1285
Classified: 666, Filtered: 619
R4 transform: _t64
Label counts:
  ANSWER_LEAKED: 278
  FILTERED: 619
  ILLEGIBLE: 288
  REASONING_LEGIBLE: 100

Per-generator:
  G1: 219 classified (leaked=25%, legible=15%, illegible=60%)
  G2: 218 classified (leaked=69%, legible=17%, illegible=14%)
  G3: 229 classified (leaked=32%, legible=13%, illegible=55%)
  G1 within-Q pairs: 0
  G3 within-Q pairs: 0
Total classified CoTs with text: 666
R5 crossfill inputs: 666
R5 C2 success rate: 59.9%
  ANSWER_LEAKED: 278 samples, R5 success=68.7%
  REASONING_LEGIBLE: 100 samples, R5 success=76.0%
  ILLEGIBLE: 288 samples, R5 success=45.8%


In [ ]:
# Check if all R5 outputs already exist (skip model load if so)
r5_outputs = ["R5_last_token", "R5_cot_boundary"]
r5_all_cached = all((ACTIVATIONS_DIR / name / "metadata.json").exists() for name in r5_outputs)

if r5_all_cached:
    print("CACHED: All R5 outputs exist, skipping R5 model load")
    model_r5 = tok_r5 = None
    n_layers_r5 = None
else:
    # Load R5 (Gemma-4-31B-IT) at full bf16 -- fits on H200 without quantization
    model_r5, tok_r5 = load_model("R5")
    from phase2_utils import _get_num_hidden_layers
    n_layers_r5 = _get_num_hidden_layers(model_r5)
    print(f"R5 layers: {n_layers_r5}")

Loading R5 (google/gemma-4-31b-it)...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]

In [ ]:
output_dir = ACTIVATIONS_DIR / "R5_last_token"
if (output_dir / "metadata.json").exists():
    print(f"CACHED: {output_dir} already exists, skipping extraction")
    r5_last_token = load_activations(output_dir)
else:
    # Extract last-token activations at every 4th layer
    layer_indices = get_default_layer_indices(n_layers_r5, stride=4)
    print(f"Extracting at layers: {layer_indices}")

    r5_last_token = extract_activations(
        model_r5, tok_r5, r5_inputs,
        layer_indices=layer_indices,
        pooling="last_token",
        batch_size=1,
        max_length=16384,
    )

    # Verify shapes
    for layer_idx, acts in r5_last_token.items():
        assert acts.shape[0] == len(r5_inputs)
        assert acts.shape[1] == LOCAL_MODELS["R5"]["hidden_dim"]
        assert not np.any(np.isnan(acts))
        assert not np.any(np.isinf(acts))
        print(f"  Layer {layer_idx}: shape={acts.shape}, norm={np.linalg.norm(acts, axis=1).mean():.2f}")

    metadata_r5 = {
        "model": "R5",
        "hf_id": LOCAL_MODELS["R5"]["hf_id"],
        "pooling": "last_token",
        "quantization": "none_bf16",
        "n_samples": len(r5_inputs),
        "layer_indices": layer_indices,
        "sample_metadata": r5_metadata,
        "labels": r5_labels,
    }
    save_activations(r5_last_token, output_dir, metadata=metadata_r5)

In [ ]:
output_dir = ACTIVATIONS_DIR / "R5_cot_boundary"
if (output_dir / "metadata.json").exists():
    print(f"CACHED: {output_dir} already exists, skipping extraction")
    r5_boundary = load_activations(output_dir)
else:
    # Extract activation at the CoT/generation boundary
    # This is the position just after the </think> tag, where the reader
    # transitions from reading the CoT to generating its answer.
    from tqdm import tqdm

    layer_indices = get_default_layer_indices(n_layers_r5, stride=4)
    boundary_activations = {idx: [] for idx in layer_indices}

    for i in tqdm(range(len(r5_inputs)), desc="Extracting boundary activations"):
        text = r5_inputs[i]
        inputs = tok_r5(
            text, return_tensors="pt", truncation=True, max_length=16384
        ).to(next(model_r5.parameters()).device)

        # Find </think> boundary token position
        think_end_tokens = tok_r5.encode("</think>", add_special_tokens=False)
        input_ids = inputs["input_ids"][0].tolist()
        boundary_pos = None
        for j in range(len(input_ids) - len(think_end_tokens) + 1):
            if input_ids[j:j + len(think_end_tokens)] == think_end_tokens:
                boundary_pos = j + len(think_end_tokens)
                break

        if boundary_pos is None:
            # Fallback: use position at 90% of sequence (near end of CoT)
            boundary_pos = int(len(input_ids) * 0.9)

        boundary_pos = min(boundary_pos, len(input_ids) - 1)

        with torch.no_grad():
            outputs = model_r5(**inputs, output_hidden_states=True)

        for layer_idx in layer_indices:
            if layer_idx < len(outputs.hidden_states):
                act = outputs.hidden_states[layer_idx][0, boundary_pos, :].cpu().float().numpy()
                boundary_activations[layer_idx].append(act)

        del outputs
        torch.cuda.empty_cache()

    # Stack into arrays
    r5_boundary = {
        idx: np.stack(acts, axis=0)
        for idx, acts in boundary_activations.items()
        if acts
    }

    for layer_idx, acts in r5_boundary.items():
        print(f"  Layer {layer_idx}: shape={acts.shape}")

    metadata_r5_boundary = {
        "model": "R5",
        "hf_id": LOCAL_MODELS["R5"]["hf_id"],
        "pooling": "cot_boundary_position",
        "quantization": "none_bf16",
        "n_samples": len(r5_inputs),
        "layer_indices": layer_indices,
        "sample_metadata": r5_metadata,
        "labels": r5_labels,
    }
    save_activations(r5_boundary, output_dir, metadata=metadata_r5_boundary)

In [ ]:
# Summary of saved files
for subdir in ['R5_last_token', 'R5_cot_boundary']:
    path = ACTIVATIONS_DIR / subdir
    if path.exists():
        total_size = sum(f.stat().st_size for f in path.rglob('*') if f.is_file())
        print(f"  {subdir}: {total_size / 1e9:.2f} GB")
    else:
        print(f"  {subdir}: NOT FOUND")

In [ ]:
# Cleanup
if model_r5 is not None:
    del model_r5, tok_r5
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory after cleanup: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
print("NB02b complete.")